In [76]:
import pandas as pd
import unicodedata

# Utils

In [77]:
budget = 500
tot_per_ruolo = {
    'Portieri': 3,
    'Difensori': 8,
    'Centrocampisti': 8,
    'Attaccanti': 6
}

# Map used to normalize the players' name
SPECIAL_MAP = {
    'ı': 'i',   # dotless i turca
    'ł': 'l',   # l barrata (polacco)
    'ø': 'o',
    'á': 'a',
    'ó': 'o',
    'ž': 'z',
    'ć': 'c',
    '-': ' '
}

# Columns in the datasets
cols24 = "Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,Min,90s,Gls,Ast,G+A,G-PK,PK,PKatt,CrdY,CrdR,xG,npxG,xAG,npxG+xAG,PrgC,PrgP,PrgR,G+A-PK,xG+xAG,Rk_stats_shooting,Nation_stats_shooting,Pos_stats_shooting,Comp_stats_shooting,Age_stats_shooting,Born_stats_shooting,90s_stats_shooting,Gls_stats_shooting,Sh,SoT,SoT%,Sh/90,SoT/90,G/Sh,G/SoT,Dist,FK,PK_stats_shooting,PKatt_stats_shooting,xG_stats_shooting,npxG_stats_shooting,npxG/Sh,G-xG,np:G-xG,Rk_stats_passing,Nation_stats_passing,Pos_stats_passing,Comp_stats_passing,Age_stats_passing,Born_stats_passing,90s_stats_passing,Cmp,Att,Cmp%,TotDist,PrgDist,Ast_stats_passing,xAG_stats_passing,xA,A-xAG,KP,1/3,PPA,CrsPA,PrgP_stats_passing,Rk_stats_passing_types,Nation_stats_passing_types,Pos_stats_passing_types,Comp_stats_passing_types,Age_stats_passing_types,Born_stats_passing_types,90s_stats_passing_types,Att_stats_passing_types,Live,Dead,FK_stats_passing_types,TB,Sw,Crs,TI,CK,In,Out,Str,Cmp_stats_passing_types,Off,Blocks,Rk_stats_gca,Nation_stats_gca,Pos_stats_gca,Comp_stats_gca,Age_stats_gca,Born_stats_gca,90s_stats_gca,SCA,SCA90,PassLive,PassDead,TO,Sh_stats_gca,Fld,Def,GCA,GCA90,Rk_stats_defense,Nation_stats_defense,Pos_stats_defense,Comp_stats_defense,Age_stats_defense,Born_stats_defense,90s_stats_defense,Tkl,TklW,Def 3rd,Mid 3rd,Att 3rd,Att_stats_defense,Tkl%,Lost,Blocks_stats_defense,Sh_stats_defense,Pass,Int,Tkl+Int,Clr,Err,Rk_stats_possession,Nation_stats_possession,Pos_stats_possession,Comp_stats_possession,Age_stats_possession,Born_stats_possession,90s_stats_possession,Touches,Def Pen,Def 3rd_stats_possession,Mid 3rd_stats_possession,Att 3rd_stats_possession,Att Pen,Live_stats_possession,Att_stats_possession,Succ,Succ%,Tkld,Tkld%,Carries,TotDist_stats_possession,PrgDist_stats_possession,PrgC_stats_possession,1/3_stats_possession,CPA,Mis,Dis,Rec,PrgR_stats_possession,Rk_stats_playing_time,Nation_stats_playing_time,Pos_stats_playing_time,Comp_stats_playing_time,Age_stats_playing_time,Born_stats_playing_time,MP_stats_playing_time,Min_stats_playing_time,Mn/MP,Min%,90s_stats_playing_time,Starts_stats_playing_time,Mn/Start,Compl,Subs,Mn/Sub,unSub,PPM,onG,onGA,+/-,+/-90,On-Off,onxG,onxGA,xG+/-,xG+/-90,Rk_stats_misc,Nation_stats_misc,Pos_stats_misc,Comp_stats_misc,Age_stats_misc,Born_stats_misc,90s_stats_misc,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld_stats_misc,Off_stats_misc,Crs_stats_misc,Int_stats_misc,TklW_stats_misc,PKwon,PKcon,OG,Recov,Won,Lost_stats_misc,Won%,Rk_stats_keeper,Nation_stats_keeper,Pos_stats_keeper,Comp_stats_keeper,Age_stats_keeper,Born_stats_keeper,MP_stats_keeper,Starts_stats_keeper,Min_stats_keeper,90s_stats_keeper,GA,GA90,SoTA,Saves,Save%,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Rk_stats_keeper_adv,Nation_stats_keeper_adv,Pos_stats_keeper_adv,Comp_stats_keeper_adv,Age_stats_keeper_adv,Born_stats_keeper_adv,90s_stats_keeper_adv,GA_stats_keeper_adv,PKA_stats_keeper_adv,FK_stats_keeper_adv,CK_stats_keeper_adv,OG_stats_keeper_adv,PSxG,PSxG/SoT,PSxG+/-,/90,Cmp_stats_keeper_adv,Att_stats_keeper_adv,Cmp%_stats_keeper_adv,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist"
cols25 = "Id,R,RM,Nome,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M"
cols24 = cols24.split(',')
cols25 = cols25.split(',')
unused_cols = ['RM', 'Qt.A M', 'Qt.I M', 'Diff.M', 'FVM M']
relevant_cols24 = [
    "Player","Nation","Pos","Squad","Age","Born",
    "MP","Starts","Min","90s","Gls","Ast","G+A","G-PK","PK","PKatt",
    "xG","xAG","npxG","npxG+xAG",
    "Sh","SoT","SoT%","Sh/90","SoT/90",
    "Cmp","Att","Cmp%","PrgP","KP","PPA",
    "PrgC","PrgR","SCA","SCA90","GCA","GCA90",
    "Tkl","Int","Blocks","Tkl+Int","Clr","Err",
    "Touches","Carries","Mis","Dis",
    "CrdY","CrdR",
    # GK fields
    "MP_stats_keeper","Starts_stats_keeper","Min_stats_keeper","90s_stats_keeper",
    "GA","GA90","SoTA","Saves","Save%","W","D","L","CS","CS%",
    "PKA","PKsv","PKm","PSxG","PSxG/SoT","PSxG+/-","/90"
]

# Weights for the teams
teams = {
    'Atalanta': {'valutazione': 80, 'gol_fatti': 78, 'gol_subiti': 37},
    'Bologna': {'valutazione': 70, 'gol_fatti': 57, 'gol_subiti': 47},
    'Cagliari': {'valutazione': 30, 'gol_fatti': 40, 'gol_subiti': 56},
    'Como': {'valutazione': 60, 'gol_fatti': 49, 'gol_subiti': 52},
    'Cremonese': {'valutazione': 35,'gol_fatti': 40, 'gol_subiti': 60}, # dati presupposti
    'Fiorentina': {'valutazione': 75, 'gol_fatti': 60, 'gol_subiti': 41},
    'Genoa': {'valutazione': 25, 'gol_fatti': 37, 'gol_subiti': 49,},
    'Juventus': {'valutazione': 90,'gol_fatti': 58, 'gol_subiti': 35},
    'Inter': {'valutazione': 100, 'gol_fatti': 79, 'gol_subiti': 35},
    'Lazio': {'valutazione': 80, 'gol_fatti': 61, 'gol_subiti': 49},
    'Lecce': {'valutazione': 20, 'gol_fatti': 27, 'gol_subiti': 58},
    'Milan': {'valutazione': 95, 'gol_fatti': 61, 'gol_subiti': 43},
    'Napoli': {'valutazione': 100, 'gol_fatti': 59, 'gol_subiti': 27},
    'Parma': {'valutazione': 40, 'gol_fatti': 44, 'gol_subiti': 58},
    'Pisa': {'valutazione': 30, 'gol_fatti': 40, 'gol_subiti': 50,}, # dati presupposti
    'Roma': {'valutazione': 80, 'gol_fatti': 56, 'gol_subiti': 35},
    'Sassuolo': {'valutazione': 30, 'gol_fatti': 45, 'gol_subiti': 55}, # dati presupposti
    'Torino': {'valutazione': 60, 'gol_fatti': 39, 'gol_subiti': 45},
    'Udinese': {'valutazione': 50, 'gol_fatti': 41, 'gol_subiti': 56},
    'Verona': {'valutazione': 10, 'gol_fatti': 34, 'gol_subiti': 66}
}

# ------------------------- Methods ------------------------- #

def eval_player(player: pd.DataFrame, team: dict) -> tuple:
    score = score_with_team = 0.0
    # Switch-case on player role
    if player['R'] == 'P':
        score = ((player['90s_stats_keeper']/38) * 0.50) + (player['CS%'] * 0.20) + (player['Save%'] * 0.10) + (player['Qt.A']/500)
        #print(f"score = {score} = {((player['90s_stats_keeper']/38) * 0.50)} + {(player['CS%'] * 0.20)} + {(player['Save%'] * 0.10)} + {(player['Qt.A']/500)}")
        score_with_team = score - score * 0.40 * (1 - (team['valutazione']/100)) - score * 0.40 * (team['gol_subiti']/max_team_gol_subiti)
    elif player['R'] == 'D':
        score = int('MF' in player['Pos']) * 10 + int('FW' in player['Pos']) * 20 + player['Gls'] * 3 + player['Ast'] * 2 + player['Tkl+Int'] * 0.50 + player['Diff.'] * 3 + player['Qt.A']
        score -= player['Err'] * 2 + player['CrdY'] * 0.50 + player['CrdR'] * 2
        score_with_team = score + score * 0.25 * (team['valutazione']/100) - score * 0.25 * (team['gol_subiti']/max_team_gol_subiti) + score * 0.10 * (team['gol_fatti']/100)
    elif player['R'] == 'C':
        score = int('FW' in player['Pos']) * 30 + player['Gls'] * 3 + player['Ast'] * 2 + player['Tkl+Int'] * 0.50 + player['Diff.'] * 3 + player['Qt.A']
        score -= player['Err'] * 2 + player['CrdY'] * 0.50 + player['CrdR'] * 2
        score_with_team = score + score * 0.30 * (team['valutazione']/100) + score * 0.20 * (team['gol_fatti']/100)
    elif player['R'] == 'A':
        score = player['Gls'] * 3 + player['Ast'] * 2 + player['Diff.'] + player['Qt.A']
        score -= player['Err'] * 2 + player['CrdY'] * 0.50 + player['CrdR'] * 2
        score_with_team = score + score * 0.30 * (team['valutazione']/100) - score * 0.20 * (team['gol_subiti']/max_team_gol_subiti) + score * 0.20 * (team['gol_fatti']/100)
    return score, score_with_team

def load_data(path: str, delimiter=' ', columns_name=[]) -> pd.DataFrame:
    '''
    Returns: the DataFrame associated to the Data set found at path \"path\".

    Parameters:
    - path: path to the CSV file with data.\n
    - delimiter: character used as delimiter in the CSV file.
    '''
    return pd.read_csv(filepath_or_buffer=path, names=columns_name, delimiter=delimiter, comment='#')

def normalize_name(s: str) -> str:
    ''''
    Returns: a normalized version of the string s.

    Parameters:
    - s: string to normalize.
    '''
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()

    # Substitutions of special characters
    for k, v in SPECIAL_MAP.items():
        s = s.replace(k, v)

    # Normalization in NFKD
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))

    s = ''.join(substr+' ' for substr in s.split(' ') if '.' not in substr)

    return s[:-1]

def get_players_by_role(df: pd.DataFrame, roles: list[str]):
    return df[df['R'].isin(roles)].copy()

def normalize_scores(df: pd.DataFrame):
    min_score = df['PlayerScore'].min()
    max_score = df['PlayerScore'].max() - min_score
    df['PlayerScore'] += -min_score
    df['PlayerScore'] = (df['PlayerScore'] / max_score) * 100.0
    min_score = df['Score'].min()
    max_score = df['Score'].max() - min_score
    df['Score'] += -min_score
    df['Score'] = (df['Score'] / max_score) * 100.0
    df.sort_values(by='Score', ascending=False, inplace=True)
    return df

# Data loading + Normalization

In [78]:
players2024: pd.DataFrame = load_data(
    path='csv/players_24-25.csv',
    delimiter=',',
    columns_name=cols24
)

players2024

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,2004.0,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,2006.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2849,2850,Edhy Zuliani,fr FRA,DF,Toulouse,fr Ligue 1,19.0,2004.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2850,2851,Szymon Żurkowski,pl POL,MF,Empoli,it Serie A,26.0,1997.0,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2851,2852,Martin Ødegaard,no NOR,MF,Arsenal,eng Premier League,25.0,1998.0,30,26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2852,2853,Milan Đurić,ba BIH,FW,Monza,it Serie A,34.0,1990.0,18,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
players2025: pd.DataFrame = load_data(
    path='csv/players_25-26.csv',
    delimiter=',',
    columns_name=cols25
)

players2025

,Id,R,RM,Nome,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M
0,5876,P,Por,Di Gregorio,Juventus,17,16,1,17,16,1,87,87
1,2428,P,Por,Sommer,Inter,16,16,0,16,16,0,86,86
2,4312,P,Por,Maignan,Milan,16,16,0,16,16,0,79,79
3,572,P,Por,Meret,Napoli,16,16,0,16,16,0,95,95
4,5841,P,Por,Svilar,Roma,16,15,1,16,15,1,85,85
...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,7158,A,Pc,Moro L.,Sassuolo,1,1,0,1,1,0,1,1
529,7159,A,Pc,Skjellerup,Sassuolo,1,1,0,1,1,0,1,1
530,6827,A,A,Njie,Torino,1,1,0,1,1,0,1,1
531,7264,A,Pc,Ambrosino,Napoli,1,1,0,1,1,0,1,1


# Dataset merge
Copy relevant information from dataset 2024 to dataset 2025.

In [80]:
# Players' name normalization
players2025['Nome'] = players2025['Nome'].apply(normalize_name)
players2024['Player'] = players2024['Player'].apply(normalize_name)

# Initialize the new empty columns
for col in relevant_cols24:
    players2025[col] = None

# Add player information into the new columns (from dataset 2024 to dataset 2025)
n_prints = 10
for i, player_name in enumerate(players2025['Nome']):
    mask = players2024['Player'].str.contains(player_name)
    if mask.sum() >= 1:
        player = players2024[mask].iloc[0]
        for col in relevant_cols24:
            players2025.at[i, col] = player[col]
        if n_prints > 0:
            #print(f'Matched: {player_name} -> {row24["Player"]}\n\n')
            n_prints -= 1

# Remove useless columns
players2025 = players2025.drop(columns=unused_cols)

print(players2025.columns)
players2024[players2024['Player'].str.contains('yildiz', na=False)]

Index(['Id', 'R', 'Nome', 'Squadra', 'Qt.A', 'Qt.I', 'Diff.', 'FVM', 'Player',
       'Nation', 'Pos', 'Squad', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s',
       'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'xG', 'xAG', 'npxG',
       'npxG+xAG', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'Cmp', 'Att',
       'Cmp%', 'PrgP', 'KP', 'PPA', 'PrgC', 'PrgR', 'SCA', 'SCA90', 'GCA',
       'GCA90', 'Tkl', 'Int', 'Blocks', 'Tkl+Int', 'Clr', 'Err', 'Touches',
       'Carries', 'Mis', 'Dis', 'CrdY', 'CrdR', 'MP_stats_keeper',
       'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA',
       'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKA',
       'PKsv', 'PKm', 'PSxG', 'PSxG/SoT', 'PSxG+/-', '/90'],
      dtype='object')


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
2799,2800,kenan yildiz,tr TUR,"FW,MF",Juventus,it Serie A,19.0,2005.0,35,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Evaluation phase

In [81]:
# Search the maximum number of gol recieved
max_team_gol_subiti = 0
for _, team_stats in teams.items():
    if team_stats['gol_subiti'] > max_team_gol_subiti:
        max_team_gol_subiti = team_stats['gol_subiti']

# Evaluation of the players
players_eval = players2025.copy()
players_eval['PlayerScore'] = 0.0
players_eval['Score'] = 0.0
for i, player in players_eval.iterrows():
    #print(f"{player['Nome']} {player['Squadra']}:")
    #if i > 10:
    #    break
    try:
        score, player_score = eval_player(player, teams[player['Squadra']])
        players_eval.at[i, 'PlayerScore'] = player_score
        players_eval.at[i, 'Score'] = score
    except Exception:
        players_eval.at[i, 'PlayerScore'] = 0.0
        players_eval.at[i, 'Score'] = 0.0

players_eval[players_eval['Score'] > 0].sort_values(by='PlayerScore', ascending=False).head(20)

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
256,2423,C,pulisic,Milan,28,28,0,218,christian pulisic,us USA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.799500,128.5
255,2167,C,orsolini,Bologna,30,30,0,218,riccardo orsolini,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,161.528000,122.0
257,632,C,zaccagni,Lazio,26,26,0,206,mattia zaccagni,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,158.673000,116.5
263,6875,C,paz,Como,21,20,1,132,nicolas paz,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,154.638000,121.0
261,2379,C,rabiot,Milan,23,23,0,145,adrien rabiot,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,128.037000,91.0
260,2517,C,de bruyne,Napoli,23,23,0,139,kevin de bruyne,be BEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,124.784000,88.0
284,22,C,de roon,Atalanta,12,12,0,32,marten de roon,nl NED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122.848000,88.0
287,4892,C,saelemaekers,Milan,12,12,0,40,alexis saelemaekers,be BEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.002000,86.0
438,2531,A,lukaku,Napoli,30,31,-1,120,romelu lukaku,be BEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.920182,89.0
268,536,C,politano,Napoli,16,15,1,65,matteo politano,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.403000,83.5


# Players

In [82]:
players_dca_norm = get_players_by_role(players_eval, ['D','C','A'])
players_dca_norm = normalize_scores(players_eval)
players_dca_norm.to_csv('csv/players_dca_out.csv', index=False)

players_dca_norm

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
256,2423,C,pulisic,Milan,28,28,0,218,christian pulisic,us USA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000
255,2167,C,orsolini,Bologna,30,30,0,218,riccardo orsolini,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.420205,95.000000
263,6875,C,paz,Como,21,20,1,132,nicolas paz,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.637688,94.230769
257,632,C,zaccagni,Lazio,26,26,0,206,mattia zaccagni,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.852849,90.769231
250,6431,D,paz,Sassuolo,1,1,0,5,nicolas paz,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.791793,76.538462
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,6184,P,martinelli,Fiorentina,1,1,0,1,gabriel martinelli,br BRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,219,P,sommariva,Genoa,1,1,0,1,daniele sommariva,it ITA,...,NaN,0.0,0.0,0.0,0.9,0.3,-0.1,-0.36,NaN,NaN
43,2815,P,terracciano,Milan,1,1,0,1,filippo terracciano,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,2271,P,nicolas,Pisa,1,1,0,1,nicolas cozza,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
portieri = get_players_by_role(players_eval, ['P'])
portieri.to_csv('csv/players_portieri_out.csv', index=False)

portieri

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
54,543,P,padelli,Udinese,1,1,0,1,daniele padelli,it ITA,...,100.0,0.0,0.0,0.0,0.8,0.26,0.8,0.77,8.333332,24.242429
28,4,P,sportiello,Atalanta,1,1,0,1,marco sportiello,it ITA,...,100.0,0.0,0.0,0.0,2.2,0.36,0.2,0.12,10.938218,21.684939
38,218,P,perin,Juventus,1,1,0,1,mattia perin,it ITA,...,60.0,0.0,0.0,0.0,3.7,0.23,0.7,0.14,9.036218,16.690607
3,572,P,meret,Napoli,16,16,0,95,alex meret,it ITA,...,47.1,1.0,2.0,1.0,25.3,0.27,2.3,0.07,8.563545,14.254980
4,5841,P,svilar,Roma,16,15,1,85,mile svilar,rs SRB,...,42.1,4.0,0.0,0.0,42.5,0.26,8.5,0.22,7.311889,14.155385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34,6184,P,martinelli,Fiorentina,1,1,0,1,gabriel martinelli,br BRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,219,P,sommariva,Genoa,1,1,0,1,daniele sommariva,it ITA,...,NaN,0.0,0.0,0.0,0.9,0.3,-0.1,-0.36,NaN,NaN
43,2815,P,terracciano,Milan,1,1,0,1,filippo terracciano,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,2271,P,nicolas,Pisa,1,1,0,1,nicolas cozza,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
difensori = get_players_by_role(players_eval, ['D'])
difensori = normalize_scores(difensori)
difensori.to_csv('csv/players_difensori_out.csv', index=False)

difensori

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
250,6431,D,paz,Sassuolo,1,1,0,5,nicolas paz,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.864677,100.000000
245,7164,D,perez,Lecce,1,1,0,1,ayoze perez,es ESP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.970855,99.497487
199,7202,D,ndiaye,Parma,4,4,0,10,iliman ndiaye,sn SEN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.883207,85.929648
62,5513,D,dumfries,Inter,21,20,1,107,denzel dumfries,nl NED,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,82.412060
63,254,D,dimarco,Inter,19,19,0,94,federico dimarco,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.162485,80.904523
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,7154,D,odenthal,Sassuolo,3,3,0,8,None,None,...,None,None,None,None,None,None,None,None,1.386677,1.507538
222,6041,D,schuurs,Torino,3,3,0,5,None,None,...,None,None,None,None,None,None,None,None,1.386677,1.507538
243,2847,D,sernicola,Cremonese,1,1,0,2,None,None,...,None,None,None,None,None,None,None,None,1.386677,1.507538
253,7255,D,britschgi,Parma,1,1,0,2,None,None,...,None,None,None,None,None,None,None,None,1.386677,1.507538


In [85]:
centrocampisti = get_players_by_role(players_eval, ['C'])
centrocampisti = normalize_scores(centrocampisti)
centrocampisti.to_csv('csv/players_centrocampisti_out.csv', index=False)

centrocampisti

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
256,2423,C,pulisic,Milan,28,28,0,218,christian pulisic,us USA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000
255,2167,C,orsolini,Bologna,30,30,0,218,riccardo orsolini,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.340955,94.941634
263,6875,C,paz,Como,21,20,1,132,nicolas paz,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.530104,94.163424
257,632,C,zaccagni,Lazio,26,26,0,206,mattia zaccagni,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.761858,90.661479
261,2379,C,rabiot,Milan,23,23,0,145,adrien rabiot,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.817121,70.817121
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,7145,C,piccinini,Pisa,6,6,0,9,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
354,7254,C,lorran,Pisa,7,7,0,15,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
350,5844,C,thorstvedt,Sassuolo,7,8,-1,22,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
305,5561,C,stengs,Pisa,11,11,0,35,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000


In [86]:
attaccanti = get_players_by_role(players_eval, ['A'])
attaccanti = normalize_scores(attaccanti)
attaccanti.to_csv('csv/players_attaccanti_out.csv', index=False)

attaccanti

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,CS%,PKA,PKsv,PKm,PSxG,PSxG/SoT,PSxG+/-,/90,PlayerScore,Score
438,2531,A,lukaku,Napoli,30,31,-1,120,romelu lukaku,be BEL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.000000,100.000000
436,2097,A,kean,Fiorentina,31,33,-2,364,moise kean,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.662622,91.573034
439,4730,A,lookman,Atalanta,28,29,-1,123,ademola lookman,ng NGA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,84.209883,87.640449
454,6435,A,krstovic,Atalanta,20,20,0,148,nikola krstovic,me MNE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.776833,67.415730
446,4510,A,leao,Milan,23,24,-1,210,rafael leao,pt POR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.877694,66.853933
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525,2012,A,milik,Juventus,1,1,0,5,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
520,7162,A,giovane,Verona,3,1,2,20,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
519,505,A,bonazzoli,Cremonese,3,1,2,19,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000
527,7171,A,buffon,Pisa,1,1,0,1,None,None,...,None,None,None,None,None,None,None,None,0.000000,0.000000


# Searching for betting

In [87]:
bet_players = pd.DataFrame(columns=players_dca_norm.columns)
for i, player in players_dca_norm.iterrows():
    if player['Qt.I'] < 5 and player['Player'] is not None and player['Min']/38 > 0.75 and \
        ((player['Score'] > 0 and player['Qt.I'] / player['Score'] > 0.3) or (player['PlayerScore'] > 0 and player['Qt.I'] / player['PlayerScore'] > 0.3)):
        bet_players = pd.concat([bet_players, pd.DataFrame([player])], ignore_index=True)
bet_players = bet_players.drop(columns=['CS%', 'PKA', 'PKsv', 'PKm', 'PSxG', 'PSxG+/-', 'PSxG+/-', '/90'])
bet_players.sort_values(by='Score', ascending=False, inplace=True)
bet_players

,Id,R,Nome,Squadra,Qt.A,Qt.I,Diff.,FVM,Player,Nation,...,SoTA,Saves,Save%,W,D,L,CS,PSxG/SoT,PlayerScore,Score
0,5532,D,bradaric,Verona,4,4,0,8,domagoj bradaric,hr CRO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.957081,22.307692
1,6957,D,valentini,Verona,4,4,0,9,nicolas valentini,ar ARG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.624691,20.000000
2,7010,C,al musrati,Verona,4,4,0,9,almoatasem al musrati,ly LBY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.895067,15.384615
3,5899,D,gigot,Lazio,4,4,0,7,samuel gigot,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.075097,14.615385
4,487,D,de silvestri,Bologna,4,4,0,10,lorenzo de silvestri,it ITA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.579961,14.230769
5,5435,D,moreno,Como,3,3,0,7,alberto moreno,es ESP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.820172,13.846154
6,6482,P,mandas,Lazio,3,3,0,3,christos mandas,gr GRE,...,27.0,20.0,74.1,3.0,5.0,1.0,4.0,0.29,6.357813,13.780324
7,7028,C,bernede,Verona,4,4,0,10,antoine bernede,fr FRA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.388099,13.461538
8,4284,C,sergi roberto,Como,4,4,0,8,sergi roberto,es ESP,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.618377,13.076923
9,5784,C,aebischer,Pisa,4,4,0,14,michel aebischer,ch SUI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.699374,13.076923


# Asta live

In [88]:
players_bought = {
    'Alessandro': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Bocco': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Corso': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Delgu': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Fabio': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Francesco': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Michele': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Panza': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Silvano': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    },
    'Varro': {
        'Portieri': [(),(),()],
        'Difensori': [(),(),(),(),(),(),(),()],
        'Centrocampisti': [(),(),(),(),(),(),(),()],
        'Attaccanti': [(),(),(),(),(),()]
    }
}

spesa_per_ruolo = {}
for key in players_bought.keys():
    spesa_per_ruolo[key] = {'Portieri': 0, 'Difensori': 0, 'Centrocampisti': 0, 'Attaccanti': 0, 'Spesi': 0, 'Crediti': 0}

# Compute the cost of the players bought
for nome, acquisti in players_bought.items():
    for ruolo, giocatori in acquisti.items():
        spesa = 0
        n_players = 0
        try:
            for giocatore, prezzo in giocatori:
                spesa += prezzo
                n_players += 1
        except:
            pass
        spesa_per_ruolo[nome][ruolo] = f'{spesa} ({n_players}/{tot_per_ruolo[ruolo]})'
        spesa_per_ruolo[nome]['Spesi'] += spesa
    spesa_per_ruolo[nome]['Crediti'] = budget - spesa_per_ruolo[nome]['Spesi']

# Create the DataFrame for the players bought
df_spesa = pd.DataFrame(spesa_per_ruolo).T
df_spesa

,Portieri,Difensori,Centrocampisti,Attaccanti,Spesi,Crediti
Alessandro,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Bocco,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Corso,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Delgu,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Fabio,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Francesco,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Michele,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500
Panza,40 (1/3),0 (0/8),0 (0/8),0 (0/6),40,460
Silvano,0 (0/3),0 (0/8),320 (8/8),0 (0/6),320,180
Varro,0 (0/3),0 (0/8),0 (0/8),0 (0/6),0,500


In [89]:
for nome, acquisti in players_bought.items():
    print(f'\n--- {nome} ---')
    for ruolo, giocatori in acquisti.items():
        print(f'{ruolo}:')
        prezzo_tot = 0
        try:
            for giocatore, prezzo in giocatori:
                prezzo_tot += prezzo
                print(f'  - {giocatore}: {prezzo}')
        except:
            pass
        print(f'    Tot: {prezzo_tot}')


--- Alessandro ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Bocco ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Corso ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Delgu ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Fabio ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Francesco ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Michele ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Panza ---
Portieri:
  - scezny: 40
    Tot: 40
Difensori:
    Tot: 0
Centrocampisti:
    Tot: 0
Attaccanti:
    Tot: 0

--- Silvano ---
Portieri:
    Tot: 0
Difensori:
    Tot: 0
Centrocampisti:
  - scezny: 40
  - scezny: 40
  -

In [90]:
players2024[players2024['Player'].str.contains('diz', na=False)]

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
2799,2800,kenan yildiz,tr TUR,"FW,MF",Juventus,it Serie A,19.0,2005.0,35,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Budget management

### Portieri (40):
- 29
- 10
- 1

### Difensori (78):
- 40
- 20
- 10
- 2
- 2
- 2
- 1
- 1

### Centrocampisti (190):
- 85
- 70
- 20
- 6
- 5
- 3
- 1
- 1

o

- 85
- 55
- 40
- 10
- 6
- 3
- 1
- 1

### Attaccanti (192):
- 130
- 40
- 18
- 2
- 1
- 1

o

- 90
- 80
- 18
- 2
- 1
- 1